# 🎬 Module 12.4 — RL for Video Frame Enhancement

*Reinforcement Learning for Image Processing — Real-World Projects*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_12_Real_World_Projects/04_Video_Frame_Enhancement/video_frame_enhancement.ipynb)

---

## Learning Objectives

By the end of this notebook you will be able to:

1. **Formulate** video frame enhancement as a sequential MDP with temporal context
2. **Explain** why temporal consistency is critical and how RL naturally addresses it
3. **Design** a reward function that balances per-frame quality, temporal smoothness, and action coherence
4. **Implement** a recurrent DQN agent with frame-buffer memory for sequential video processing
5. **Generate** synthetic degraded video sequences with controllable temporal artefacts
6. **Train and evaluate** the agent, measuring both PSNR improvement and flickering reduction
7. **Visualise** filmstrip views, action heatmaps, and temporal consistency curves

---

## 1 — Introduction: From Images to Video

### 1.1 Video Enhancement ≠ Repeated Image Enhancement

Applying a single-image enhancement model independently to every frame of a video leads to
**temporal flickering** — jarring brightness jumps, colour shifts, and noise-pattern changes
between consecutive frames. The human visual system is extremely sensitive to such
inconsistencies, even when each individual frame looks acceptable.

### 1.2 Why RL Is a Natural Fit

Video frames arrive **sequentially**; enhancement decisions at time $t$ affect the perception of
future frames. This is precisely the setting modelled by a Markov Decision Process (MDP):

| Concept | Video Enhancement Analogue |
|---------|---------------------------|
| Episode | One video sequence |
| Time step $t$ | Frame index |
| State $s_t$ | Current frame + temporal context |
| Action $a_t$ | Enhancement operation |
| Reward $r_t$ | Frame quality + temporal consistency |

An RL agent **remembers** what it did to the previous frame and can therefore maintain
**smooth, consistent** enhancements across the whole sequence.

## 2 — Mathematical Formulation

### 2.1 State Representation

The state encodes both the current frame and a summary of recent history:

$$s_t = \bigl[I_t,\; h_{t-1}\bigr]$$

where $I_t \in \mathbb{R}^{H \times W}$ is the current (grey-scale) frame and
$h_{t-1} \in \mathbb{R}^{d}$ is a hidden vector encoding previous-frame features.
In our implementation the hidden state is maintained by a **GRU** cell:

$$h_t = \text{GRU}\!\bigl(\phi(I_t),\; h_{t-1}\bigr)$$

with $\phi$ a convolutional feature extractor.

### 2.2 Action Space

We define 10 discrete enhancement operations:

| Index | Action | Effect |
|-------|--------|--------|
| 0 | `brightness_up` | $I \leftarrow \min(I + \delta_b,\, 1)$ |
| 1 | `brightness_down` | $I \leftarrow \max(I - \delta_b,\, 0)$ |
| 2 | `contrast_up` | $I \leftarrow \text{clip}\bigl(\mu + \alpha_{\uparrow}(I - \mu)\bigr)$ |
| 3 | `contrast_down` | $I \leftarrow \text{clip}\bigl(\mu + \alpha_{\downarrow}(I - \mu)\bigr)$ |
| 4 | `denoise_light` | Gaussian blur $\sigma=0.5$ |
| 5 | `denoise_heavy` | Gaussian blur $\sigma=1.2$ |
| 6 | `sharpen` | Unsharp mask |
| 7 | `temporal_smooth` | $I_t \leftarrow 0.7\,I_t + 0.3\,I_{t-1}^{\text{enh}}$ |
| 8 | `color_correct` | Histogram-match to previous enhanced frame |
| 9 | `no_op` | Identity — keep frame unchanged |

### 2.3 Reward Design

The reward at time $t$ has three terms.

**Per-frame quality** (higher is better):

$$R_{\text{quality}} = \text{PSNR}\!\bigl(I_t^{\text{enh}},\; I_t^{\text{clean}}\bigr)$$

**Temporal consistency** (penalises content-unrelated differences between consecutive
enhanced frames):

$$R_{\text{temporal}} = -\bigl\|\,f(I_t^{\text{enh}}) - f(I_{t-1}^{\text{enh}})\bigr\|_2
  \;\cdot\;\mathbb{1}[\text{no scene change}]$$

where $f(\cdot)$ extracts a low-dimensional feature vector (we use average-pooled
$4\times4$ blocks).

**Action smoothness** (discourages erratic switching):

$$R_{\text{smooth}} = -\gamma_a \,\|a_t - a_{t-1}\|$$

where actions are treated as one-hot and the norm reduces to an indicator.

**Combined reward:**

$$R_t = \alpha \cdot R_{\text{quality}}
      + \beta  \cdot R_{\text{temporal}}
      - \gamma_a \cdot \|a_t - a_{t-1}\|$$

### 2.4 Temporal Difference Penalty for Flickering Prevention

Flickering manifests as high-frequency oscillations in per-frame brightness.
We add an explicit penalty proportional to the second-order temporal derivative
of mean frame intensity:

$$\text{Flicker}_t = \bigl|\bar{I}_t^{\text{enh}} - 2\,\bar{I}_{t-1}^{\text{enh}} + \bar{I}_{t-2}^{\text{enh}}\bigr|$$

$$R_t \;\leftarrow\; R_t - \lambda_{\text{flicker}} \cdot \text{Flicker}_t$$

This is essentially the discrete Laplacian in time, and penalising it
encourages monotonic or linear brightness trajectories.

## Deep Derivation: RL for Video Frame Enhancement and Temporal Consistency

### Step 1: Video Degradation Model
A degraded video frame:
$$I_t^{\text{deg}} = H(M_t \cdot I_t^{\text{clean}}) + n_t$$

where $M_t$ = motion-dependent blur kernel, $H$ = compression artifact operator, $n_t \sim \mathcal{N}(0, \sigma_t^2)$ is temporal noise.

**Motion blur kernel:** For camera motion $(v_x, v_y)$ during exposure time $\Delta t$:
$$M(x, y) = \frac{1}{L}\text{rect}\left(\frac{x - v_x t}{1}\right)\delta(y - v_y t), \quad L = \sqrt{v_x^2 + v_y^2}\Delta t$$

**Derivation:** The blur kernel integrates the scene along the motion trajectory:
$$I_{\text{blurred}}(x, y) = \frac{1}{\Delta t}\int_0^{\Delta t} I(x - v_x \tau, y - v_y \tau) d\tau$$

In the Fourier domain: $\hat{I}_{\text{blurred}} = \hat{M} \cdot \hat{I}$, where $\hat{M}(u,v) = \text{sinc}(v_x u + v_y v)$. The zeros of the sinc function correspond to lost frequencies — information that cannot be recovered. $\blacksquare$

### Step 2: Optical Flow for Temporal Alignment
The optical flow field $\mathbf{u} = (u, v)$ satisfies the brightness constancy constraint:
$$I(x, y, t) = I(x + u, y + v, t + 1)$$

**Derivation (Taylor expansion):**
$$I(x+u, y+v, t+1) \approx I(x,y,t) + I_x u + I_y v + I_t = I(x,y,t)$$

$$\implies I_x u + I_y v + I_t = 0 \quad \text{(optical flow constraint equation)}$$

This is one equation with two unknowns — the **aperture problem**! Need additional constraints:

**Lucas-Kanade (local smoothness):** Assume constant flow in a window $W$:
$$\min_{u,v} \sum_{(x_i,y_i) \in W} [I_x(x_i) u + I_y(x_i) v + I_t(x_i)]^2$$

Normal equations: $\begin{pmatrix} \sum I_x^2 & \sum I_x I_y \\ \sum I_x I_y & \sum I_y^2 \end{pmatrix}\begin{pmatrix} u \\ v \end{pmatrix} = -\begin{pmatrix} \sum I_x I_t \\ \sum I_y I_t \end{pmatrix}$

**Solvable when:** The structure tensor $A^T A$ is well-conditioned (both eigenvalues large → corner/texture region). $\blacksquare$

### Step 3: Temporal Consistency Loss
For video enhancement, adjacent frames must be consistent:
$$\mathcal{L}_{\text{temporal}} = \sum_t \|O_t \odot (\hat{I}_t - W(\hat{I}_{t-1}, \mathbf{u}_{t\to t-1}))\|_1$$

where $W(\cdot, \mathbf{u})$ is the warping function and $O_t$ is the occlusion mask.

**Warping via bilinear interpolation:**
$$W(I, \mathbf{u})(x, y) = I(x + u(x,y), y + v(x,y))$$

For non-integer coordinates, bilinear interpolation:
$$I(x', y') = (1-\alpha)(1-\beta)I(\lfloor x'\rfloor, \lfloor y'\rfloor) + \alpha(1-\beta)I(\lceil x'\rceil, \lfloor y'\rfloor)$$
$$+ (1-\alpha)\beta I(\lfloor x'\rfloor, \lceil y'\rceil) + \alpha\beta I(\lceil x'\rceil, \lceil y'\rceil)$$

where $\alpha = x' - \lfloor x'\rfloor$, $\beta = y' - \lfloor y'\rfloor$.

**Derivation of the occlusion mask $O_t$:** A pixel is occluded if the forward and backward flows are inconsistent:
$$O_t(x,y) = \mathbb{1}\left[\|\mathbf{u}_{t\to t-1}(x,y) + W(\mathbf{u}_{t-1\to t}, \mathbf{u}_{t\to t-1})(x,y)\|^2 < \alpha_1(|\mathbf{u}_{t\to t-1}|^2 + |W(\mathbf{u}_{t-1\to t})|^2) + \alpha_2\right]$$

This forward-backward consistency check identifies pixels where temporal warping is unreliable. $\blacksquare$

### Step 4: RL Agent for Adaptive Frame Enhancement
The agent selects enhancement parameters per-frame based on content:

**State:** $s_t = (\phi(I_t), \phi(I_{t-1}), \mathbf{u}_{t\to t-1}, q_t)$

where $q_t$ = estimated quality metrics of frame $t$.

**Action:** $a_t = (\alpha_{\text{denoise}}, \alpha_{\text{sharpen}}, \alpha_{\text{deblock}}, \alpha_{\text{color}}) \in [0,1]^4$

**Reward (temporal-aware):**
$$R_t = w_1 \cdot \Delta\text{PSNR}_t + w_2 \cdot \Delta\text{SSIM}_t - w_3 \cdot \mathcal{L}_{\text{temporal}}(t) - w_4 \cdot \text{flicker}(t)$$

**Flicker metric:**
$$\text{flicker}(t) = \|(\hat{I}_t - \hat{I}_{t-1}) - (I_t^{gt} - I_{t-1}^{gt})\|_2^2$$

This measures the difference between predicted and ground-truth temporal changes. Low flicker = temporally consistent enhancement.

### Step 5: Recurrent Policy for Temporal Modeling
Use an LSTM to maintain temporal state:
$$h_t, c_t = \text{LSTM}([\phi(I_t); a_{t-1}], h_{t-1}, c_t-1)$$
$$\pi_\theta(a_t | s_t) = \text{MLP}(h_t)$$

**Derivation of LSTM gates (complete):**
$$f_t = \sigma(W_f [h_{t-1}; x_t] + b_f) \quad \text{(forget gate)}$$
$$i_t = \sigma(W_i [h_{t-1}; x_t] + b_i) \quad \text{(input gate)}$$
$$\tilde{c}_t = \tanh(W_c [h_{t-1}; x_t] + b_c) \quad \text{(candidate)}$$
$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \quad \text{(cell state)}$$
$$o_t = \sigma(W_o [h_{t-1}; x_t] + b_o) \quad \text{(output gate)}$$
$$h_t = o_t \odot \tanh(c_t) \quad \text{(hidden state)}$$

**Why LSTM over MLP for video?** The cell state $c_t$ can maintain long-range temporal information (e.g., scene brightness, noise level) across many frames. The forget gate controls how much history to retain.

**Gradient flow:** $\frac{\partial c_t}{\partial c_{t-1}} = f_t$ (multiplicative). If $f_t \approx 1$, gradients flow unchanged — solving the vanishing gradient problem for temporal dependencies. $\blacksquare$

### Step 6: Video Quality Assessment (VMAF)
**Video Multi-Method Assessment Fusion:**
$$\text{VMAF} = \text{SVM}\left(\text{VIF}_{\text{scale}}, \text{DLM}, \text{motion}\right)$$

**Visual Information Fidelity (VIF) derivation:**
Model source as Gaussian scale mixture: $C = S \cdot U$ where $S \sim p_S$, $U \sim \mathcal{N}(0, C_U)$.

$$\text{VIF} = \frac{I(C; E | S)}{I(C; D | S)} = \frac{\sum_j \log_2(1 + \frac{g_j^2 \sigma_{u,j}^2}{\sigma_{n,j}^2})}{\sum_j \log_2(1 + \frac{\sigma_{u,j}^2}{\sigma_{v,j}^2})}$$

where $E$ = distorted, $D$ = reference, $g_j$ = gain, $\sigma_n^2$ = distortion noise, $\sigma_v^2$ = visual noise.

**Derivation:** VIF measures how much visual information (relative to a human visual system model) survives the distortion process. The ratio compares the mutual information between source and distorted vs. source and reference channels. VIF $= 1$ for perfect quality, $< 1$ for degraded. $\blacksquare$

### Step 7: Batch Processing and Look-Ahead Planning
For efficiency, the agent can plan enhancement for $N$ future frames:

**Model Predictive Control (MPC) formulation:**
$$a_{t:t+N}^* = \arg\max_{a_{t:t+N}} \sum_{k=0}^{N} \gamma^k \hat{R}_{t+k}(a_{t+k})$$

where $\hat{R}$ uses a learned dynamics model to predict future rewards.

**Derivation of computational savings:** Without look-ahead: $O(T \cdot C_{\text{enhance}})$.
With look-ahead ($N$ frame batches): $O(T/N \cdot (C_{\text{plan}} + N \cdot C_{\text{enhance}}))$.

If $C_{\text{plan}} \ll N \cdot C_{\text{enhance}}$, look-ahead adds minimal overhead but enables:
1. Pre-loading future frame features (pipeline parallelism)
2. Consistent enhancement across scene cuts
3. Optimal allocation of compute budget to harder frames

**Scene cut detection criterion:**
$$\text{scene\_cut}(t) = \mathbb{1}\left[\frac{\|I_t - I_{t-1}\|_2}{\sqrt{HW}} > \tau \text{ AND } \|\mathbf{u}_{t\to t-1}\|_\infty > \tau_u\right]$$

At scene cuts, reset the temporal state $h_t = \mathbf{0}$ to prevent artifacts from the old scene context. $\blacksquare$

## 3 — Environment Setup

In [ ]:
!pip install torch torchvision numpy matplotlib scikit-image -q

In [ ]:
# Download REAL open-source datasets for real-world projects
import torchvision
import subprocess
import sys

# MedMNIST for medical imaging (install + download)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'medmnist', '-q'])
import medmnist
from medmnist import ChestMNIST, DermaMNIST

# Chest X-rays (real medical images!)
chest_data = ChestMNIST(split='train', download=True, size=28)
print(f"✅ ChestMNIST: {len(chest_data)} real chest X-ray images")

# Dermatology images
derma_data = DermaMNIST(split='train', download=True, size=28)
print(f"✅ DermaMNIST: {len(derma_data)} real skin lesion images")

# EuroSAT for satellite imagery
try:
    eurosat = torchvision.datasets.EuroSAT(root='./data', download=True)
    print(f"✅ EuroSAT: {len(eurosat)} real satellite images (64x64, 10 land-use classes)")
except:
    print("⚠️ EuroSAT downloading...")

# CIFAR-10 for video/editing projects
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
print(f"✅ CIFAR-10: {len(cifar10)} real photos for editing/video projects")

In [ ]:
import os, math, random, collections, warnings
from typing import List, Tuple, Optional, Dict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 4 — Synthetic Video Generation

We build a generator that creates short grey-scale video clips of **moving geometric shapes**
with smooth trajectories. Each clip provides a *clean* ground-truth sequence and a
*degraded* version with temporally-varying noise, brightness drift, and random flickering.

In [ ]:
# ---------------------------------------------------------------------------
#  Synthetic video sequence generator
# ---------------------------------------------------------------------------

IMG_H, IMG_W = 64, 64
NUM_FRAMES = 20


def _draw_circle(canvas: np.ndarray, cx: float, cy: float, r: float, val: float) -> None:
    """Draw a filled circle onto *canvas* (H×W float array in [0,1])."""
    yy, xx = np.ogrid[:canvas.shape[0], :canvas.shape[1]]
    mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2
    canvas[mask] = np.clip(val, 0.0, 1.0)


def _draw_rect(canvas: np.ndarray, cx: float, cy: float, hw: float, hh: float, val: float) -> None:
    """Draw a filled axis-aligned rectangle."""
    x0, x1 = int(max(cx - hw, 0)), int(min(cx + hw, canvas.shape[1]))
    y0, y1 = int(max(cy - hh, 0)), int(min(cy + hh, canvas.shape[0]))
    canvas[y0:y1, x0:x1] = np.clip(val, 0.0, 1.0)


def generate_clean_sequence(
    num_frames: int = NUM_FRAMES,
    h: int = IMG_H,
    w: int = IMG_W,
    num_shapes: int = 3,
    rng: Optional[np.random.RandomState] = None,
) -> np.ndarray:
    """Return (num_frames, H, W) float32 array of a clean synthetic clip."""
    rng = rng or np.random.RandomState()
    bg_val = rng.uniform(0.05, 0.20)
    frames = np.full((num_frames, h, w), bg_val, dtype=np.float32)

    shapes = []
    for _ in range(num_shapes):
        kind = rng.choice(['circle', 'rect'])
        cx, cy = rng.uniform(15, w - 15), rng.uniform(15, h - 15)
        vx, vy = rng.uniform(-1.5, 1.5), rng.uniform(-1.5, 1.5)
        size = rng.uniform(5, 12)
        val = rng.uniform(0.45, 0.95)
        shapes.append(dict(kind=kind, cx=cx, cy=cy, vx=vx, vy=vy, size=size, val=val))

    for t in range(num_frames):
        for s in shapes:
            x = s['cx'] + s['vx'] * t
            y = s['cy'] + s['vy'] * t
            x = np.clip(x, s['size'], w - s['size'])
            y = np.clip(y, s['size'], h - s['size'])
            if s['kind'] == 'circle':
                _draw_circle(frames[t], x, y, s['size'], s['val'])
            else:
                _draw_rect(frames[t], x, y, s['size'], s['size'] * 0.7, s['val'])

    light_drift = np.linspace(0, rng.uniform(-0.08, 0.08), num_frames)
    for t in range(num_frames):
        frames[t] = np.clip(frames[t] + light_drift[t], 0.0, 1.0)

    return frames


def degrade_sequence(
    clean: np.ndarray,
    noise_range: Tuple[float, float] = (0.04, 0.15),
    brightness_drift: float = 0.15,
    flicker_prob: float = 0.25,
    flicker_mag: float = 0.12,
    rng: Optional[np.random.RandomState] = None,
) -> np.ndarray:
    """Apply temporally-varying degradations to a clean clip."""
    rng = rng or np.random.RandomState()
    T = clean.shape[0]
    degraded = clean.copy()

    noise_sigmas = np.linspace(*sorted(rng.uniform(*noise_range, size=2)), T)
    rng.shuffle(noise_sigmas)
    for t in range(T):
        degraded[t] += rng.randn(*degraded[t].shape).astype(np.float32) * noise_sigmas[t]

    drift = np.cumsum(rng.randn(T) * brightness_drift / T).astype(np.float32)
    for t in range(T):
        degraded[t] += drift[t]

    for t in range(T):
        if rng.rand() < flicker_prob:
            degraded[t] += rng.uniform(-flicker_mag, flicker_mag)

    degraded = np.clip(degraded, 0.0, 1.0)
    return degraded


print("Video generator ready.")

In [ ]:
# ---------------------------------------------------------------------------
#  Visualise a sample clean / degraded pair
# ---------------------------------------------------------------------------
rng_demo = np.random.RandomState(7)
demo_clean = generate_clean_sequence(rng=rng_demo)
demo_degraded = degrade_sequence(demo_clean, rng=rng_demo)

show_idxs = [0, 4, 9, 14, 19]
fig, axes = plt.subplots(2, len(show_idxs), figsize=(16, 5))
for col, idx in enumerate(show_idxs):
    axes[0, col].imshow(demo_clean[idx], cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f'Clean t={idx}', fontsize=10)
    axes[0, col].axis('off')
    axes[1, col].imshow(demo_degraded[idx], cmap='gray', vmin=0, vmax=1)
    axes[1, col].set_title(f'Degraded t={idx}', fontsize=10)
    axes[1, col].axis('off')
axes[0, 0].set_ylabel('Clean', fontsize=12)
axes[1, 0].set_ylabel('Degraded', fontsize=12)
fig.suptitle('Sample Synthetic Video Sequence', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
#  Temporal degradation analysis — mean brightness over time
# ---------------------------------------------------------------------------
clean_means = [f.mean() for f in demo_clean]
degraded_means = [f.mean() for f in demo_degraded]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(clean_means, 'g-o', label='Clean', linewidth=2, markersize=5)
ax.plot(degraded_means, 'r-s', label='Degraded', linewidth=2, markersize=5)
ax.set_xlabel('Frame Index', fontsize=12)
ax.set_ylabel('Mean Pixel Intensity', fontsize=12)
ax.set_title('Temporal Brightness Profile — Flickering Visible in Degraded', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5 — Video Enhancement Environment

The environment presents frames one at a time. The agent selects an action;
the environment applies it, computes the reward (quality + temporal consistency),
and advances to the next frame.

In [ ]:
# ---------------------------------------------------------------------------
#  Image-processing helper functions
# ---------------------------------------------------------------------------

def gaussian_blur(img: np.ndarray, sigma: float) -> np.ndarray:
    """Simple separable Gaussian blur (pure numpy)."""
    k = int(np.ceil(sigma * 3)) * 2 + 1
    ax = np.arange(-k // 2 + 1, k // 2 + 1, dtype=np.float32)
    kernel = np.exp(-0.5 * (ax / sigma) ** 2)
    kernel /= kernel.sum()
    padded = np.pad(img, k // 2, mode='reflect')
    rows = np.apply_along_axis(lambda r: np.convolve(r, kernel, mode='valid'), 1, padded)
    padded2 = np.pad(rows, ((k // 2, k // 2), (0, 0)), mode='reflect')
    result = np.apply_along_axis(lambda c: np.convolve(c, kernel, mode='valid'), 0, padded2)
    return result.astype(np.float32)


def sharpen(img: np.ndarray, amount: float = 0.6) -> np.ndarray:
    blurred = gaussian_blur(img, 1.0)
    return np.clip(img + amount * (img - blurred), 0.0, 1.0).astype(np.float32)


def compute_psnr(a: np.ndarray, b: np.ndarray) -> float:
    mse = np.mean((a - b) ** 2)
    if mse < 1e-10:
        return 50.0
    return float(10.0 * np.log10(1.0 / mse))


def frame_features(img: np.ndarray, pool: int = 4) -> np.ndarray:
    """Average-pool into pool×pool blocks → flat feature vector."""
    h, w = img.shape
    bh, bw = h // pool, w // pool
    trimmed = img[:bh * pool, :bw * pool]
    return trimmed.reshape(pool, bh, pool, bw).mean(axis=(1, 3)).ravel()


ACTION_NAMES = [
    'brightness_up', 'brightness_down',
    'contrast_up', 'contrast_down',
    'denoise_light', 'denoise_heavy',
    'sharpen', 'temporal_smooth',
    'color_correct', 'no_op',
]
NUM_ACTIONS = len(ACTION_NAMES)


class VideoEnhancementEnv:
    """Frame-sequential video enhancement environment."""

    def __init__(
        self,
        alpha: float = 0.06,
        beta: float = 2.0,
        gamma_a: float = 0.3,
        lam_flicker: float = 1.5,
        max_steps_per_frame: int = 3,
    ):
        self.alpha = alpha
        self.beta = beta
        self.gamma_a = gamma_a
        self.lam_flicker = lam_flicker
        self.max_steps_per_frame = max_steps_per_frame

        self.clean_seq: Optional[np.ndarray] = None
        self.degraded_seq: Optional[np.ndarray] = None
        self.frame_idx = 0
        self.step_in_frame = 0
        self.current_frame: Optional[np.ndarray] = None
        self.prev_enhanced: Optional[np.ndarray] = None
        self.prev_prev_enhanced: Optional[np.ndarray] = None
        self.prev_action = NUM_ACTIONS - 1  # no_op
        self.enhanced_frames: List[np.ndarray] = []
        self.action_history: List[int] = []

    def reset(self, clean_seq: np.ndarray, degraded_seq: np.ndarray) -> np.ndarray:
        self.clean_seq = clean_seq
        self.degraded_seq = degraded_seq
        self.frame_idx = 0
        self.step_in_frame = 0
        self.current_frame = degraded_seq[0].copy()
        self.prev_enhanced = None
        self.prev_prev_enhanced = None
        self.prev_action = NUM_ACTIONS - 1
        self.enhanced_frames = []
        self.action_history = []
        return self.current_frame.copy()

    def _apply_action(self, img: np.ndarray, action: int) -> np.ndarray:
        out = img.copy()
        delta_b = 0.05
        if action == 0:  # brightness_up
            out = np.clip(out + delta_b, 0, 1)
        elif action == 1:  # brightness_down
            out = np.clip(out - delta_b, 0, 1)
        elif action == 2:  # contrast_up
            mu = out.mean()
            out = np.clip(mu + 1.15 * (out - mu), 0, 1)
        elif action == 3:  # contrast_down
            mu = out.mean()
            out = np.clip(mu + 0.87 * (out - mu), 0, 1)
        elif action == 4:  # denoise_light
            out = gaussian_blur(out, 0.5)
        elif action == 5:  # denoise_heavy
            out = gaussian_blur(out, 1.2)
        elif action == 6:  # sharpen
            out = sharpen(out, 0.6)
        elif action == 7:  # temporal_smooth
            if self.prev_enhanced is not None:
                out = (0.7 * out + 0.3 * self.prev_enhanced).astype(np.float32)
        elif action == 8:  # color_correct — match mean/std to previous enhanced
            if self.prev_enhanced is not None:
                mu_prev, std_prev = self.prev_enhanced.mean(), self.prev_enhanced.std() + 1e-8
                mu_cur, std_cur = out.mean(), out.std() + 1e-8
                out = (out - mu_cur) / std_cur * std_prev + mu_prev
                out = np.clip(out, 0, 1).astype(np.float32)
        # action == 9: no_op
        return out.astype(np.float32)

    def step(self, action: int) -> Tuple[np.ndarray, float, bool, Dict]:
        self.current_frame = self._apply_action(self.current_frame, action)
        self.action_history.append(action)
        self.step_in_frame += 1

        r_quality = self.alpha * compute_psnr(self.current_frame, self.clean_seq[self.frame_idx])

        r_temporal = 0.0
        if self.prev_enhanced is not None:
            feat_cur = frame_features(self.current_frame)
            feat_prev = frame_features(self.prev_enhanced)
            r_temporal = -self.beta * np.linalg.norm(feat_cur - feat_prev)

        r_smooth = 0.0
        if action != self.prev_action:
            r_smooth = -self.gamma_a

        r_flicker = 0.0
        if self.prev_enhanced is not None and self.prev_prev_enhanced is not None:
            m_cur = self.current_frame.mean()
            m_prev = self.prev_enhanced.mean()
            m_pprev = self.prev_prev_enhanced.mean()
            r_flicker = -self.lam_flicker * abs(m_cur - 2 * m_prev + m_pprev)

        reward = r_quality + r_temporal + r_smooth + r_flicker

        advance = self.step_in_frame >= self.max_steps_per_frame
        done = False
        if advance:
            self.enhanced_frames.append(self.current_frame.copy())
            self.prev_prev_enhanced = self.prev_enhanced
            self.prev_enhanced = self.current_frame.copy()
            self.frame_idx += 1
            self.step_in_frame = 0
            if self.frame_idx >= len(self.degraded_seq):
                done = True
            else:
                self.current_frame = self.degraded_seq[self.frame_idx].copy()

        self.prev_action = action
        info = {
            'r_quality': r_quality,
            'r_temporal': r_temporal,
            'r_smooth': r_smooth,
            'r_flicker': r_flicker,
            'frame_idx': self.frame_idx,
        }
        return self.current_frame.copy(), reward, done, info


print(f"Environment ready — {NUM_ACTIONS} actions: {ACTION_NAMES}")

## 6 — Recurrent DQN Agent

The network uses a small CNN to extract spatial features from the current frame,
then feeds them through a **GRU** cell that maintains temporal context across frames.
The GRU output is mapped to Q-values for each action.

$$Q(s_t, a; \theta) = W_q \;\text{GRU}\!\bigl(\phi_{\text{CNN}}(I_t),\; h_{t-1}\bigr) + b_q$$

In [ ]:
class RecurrentDQN(nn.Module):
    """CNN + GRU → Q-values, designed for sequential frame processing."""

    def __init__(self, h: int = IMG_H, w: int = IMG_W, n_actions: int = NUM_ACTIONS, hidden: int = 128):
        super().__init__()
        self.hidden_size = hidden

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=2, padding=1),
            nn.ReLU(),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, 1, h, w)
            conv_out = self.conv(dummy)
            self.conv_flat = int(conv_out.numel())

        self.fc_pre = nn.Linear(self.conv_flat, hidden)
        self.gru = nn.GRUCell(hidden, hidden)
        self.fc_q = nn.Linear(hidden, n_actions)

    def forward(self, x: torch.Tensor, h: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """x: (B, 1, H, W), h: (B, hidden) or None.  Returns (Q-values, new_h)."""
        b = x.size(0)
        if h is None:
            h = torch.zeros(b, self.hidden_size, device=x.device)
        feat = self.conv(x).reshape(b, -1)
        feat = F.relu(self.fc_pre(feat))
        h_new = self.gru(feat, h)
        q = self.fc_q(h_new)
        return q, h_new


print("RecurrentDQN architecture:")
net_demo = RecurrentDQN().to(DEVICE)
total_params = sum(p.numel() for p in net_demo.parameters())
print(f"  Total parameters: {total_params:,}")
print(net_demo)

## 7 — Replay Buffer & Training Utilities

In [ ]:
Transition = collections.namedtuple('Transition', ('state', 'hidden', 'action', 'reward', 'next_state', 'next_hidden', 'done'))


class ReplayBuffer:
    def __init__(self, capacity: int = 20000):
        self.buffer = collections.deque(maxlen=capacity)

    def push(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size: int) -> List[Transition]:
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)


def select_action(q_net: RecurrentDQN, state: np.ndarray, hidden: torch.Tensor, epsilon: float) -> Tuple[int, torch.Tensor]:
    if random.random() < epsilon:
        action = random.randrange(NUM_ACTIONS)
        with torch.no_grad():
            s_t = torch.from_numpy(state).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
            _, h_new = q_net(s_t, hidden)
        return action, h_new
    with torch.no_grad():
        s_t = torch.from_numpy(state).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
        q_vals, h_new = q_net(s_t, hidden)
        action = int(q_vals.argmax(dim=1).item())
    return action, h_new


def optimize(q_net, target_net, optimizer, buffer, batch_size=64, gamma=0.99):
    if len(buffer) < batch_size:
        return 0.0
    transitions = buffer.sample(batch_size)
    batch = Transition(*zip(*transitions))

    states = torch.stack([torch.from_numpy(s).unsqueeze(0) for s in batch.state]).float().to(DEVICE)
    hiddens = torch.cat(batch.hidden, dim=0).to(DEVICE)
    actions = torch.tensor(batch.action, dtype=torch.long, device=DEVICE).unsqueeze(1)
    rewards = torch.tensor(batch.reward, dtype=torch.float32, device=DEVICE)
    next_states = torch.stack([torch.from_numpy(s).unsqueeze(0) for s in batch.next_state]).float().to(DEVICE)
    next_hiddens = torch.cat(batch.next_hidden, dim=0).to(DEVICE)
    dones = torch.tensor(batch.done, dtype=torch.float32, device=DEVICE)

    q_values, _ = q_net(states, hiddens)
    q_selected = q_values.gather(1, actions).squeeze(1)

    with torch.no_grad():
        q_next, _ = target_net(next_states, next_hiddens)
        q_next_max = q_next.max(dim=1)[0]
        q_target = rewards + gamma * q_next_max * (1 - dones)

    loss = F.smooth_l1_loss(q_selected, q_target)
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(q_net.parameters(), 1.0)
    optimizer.step()
    return loss.item()


print("Training utilities ready.")

## 8 — Training Loop

We train two agents in parallel:
1. **Full agent** — uses the complete reward $R_t$ (quality + temporal + smoothness + flicker)
2. **Quality-only agent** — uses only $R_{\text{quality}}$ (no temporal terms)

This lets us compare the effect of temporal-consistency reward shaping.

In [ ]:
def train_agent(
    n_episodes: int = 120,
    epsilon_start: float = 1.0,
    epsilon_end: float = 0.05,
    epsilon_decay: int = 80,
    target_update: int = 10,
    lr: float = 1e-3,
    batch_size: int = 64,
    gamma: float = 0.99,
    use_temporal_reward: bool = True,
    verbose: bool = True,
) -> Dict:
    q_net = RecurrentDQN().to(DEVICE)
    target_net = RecurrentDQN().to(DEVICE)
    target_net.load_state_dict(q_net.state_dict())
    target_net.eval()
    optimizer = optim.Adam(q_net.parameters(), lr=lr)
    buffer = ReplayBuffer(20000)

    env = VideoEnhancementEnv()
    if not use_temporal_reward:
        env.beta = 0.0
        env.gamma_a = 0.0
        env.lam_flicker = 0.0

    history = {'episode_reward': [], 'avg_psnr': [], 'avg_temporal': [],
               'loss': [], 'epsilon': [], 'actions_per_ep': []}

    rng_train = np.random.RandomState(SEED)

    for ep in range(n_episodes):
        epsilon = epsilon_end + (epsilon_start - epsilon_end) * math.exp(-ep / epsilon_decay)
        clean = generate_clean_sequence(rng=np.random.RandomState(rng_train.randint(0, 100000)))
        degraded = degrade_sequence(clean, rng=np.random.RandomState(rng_train.randint(0, 100000)))

        state = env.reset(clean, degraded)
        hidden = torch.zeros(1, q_net.hidden_size, device=DEVICE)
        ep_reward = 0.0
        ep_psnr = []
        ep_temporal = []
        ep_loss = []
        ep_actions = []
        done = False

        while not done:
            action, h_new = select_action(q_net, state, hidden, epsilon)
            next_state, reward, done, info = env.step(action)
            buffer.push(state, hidden.detach(), action, reward, next_state, h_new.detach(), done)
            state = next_state
            hidden = h_new
            ep_reward += reward
            ep_psnr.append(info['r_quality'] / env.alpha if env.alpha > 0 else 0)
            ep_temporal.append(info['r_temporal'])
            ep_actions.append(action)

            loss = optimize(q_net, target_net, optimizer, buffer, batch_size, gamma)
            if loss > 0:
                ep_loss.append(loss)

        if ep % target_update == 0:
            target_net.load_state_dict(q_net.state_dict())

        history['episode_reward'].append(ep_reward)
        history['avg_psnr'].append(np.mean(ep_psnr))
        history['avg_temporal'].append(np.mean(ep_temporal) if ep_temporal else 0)
        history['loss'].append(np.mean(ep_loss) if ep_loss else 0)
        history['epsilon'].append(epsilon)
        history['actions_per_ep'].append(ep_actions)

        if verbose and (ep + 1) % 20 == 0:
            tag = "FULL" if use_temporal_reward else "QUAL"
            print(f"[{tag}] Ep {ep+1:3d}/{n_episodes}  "
                  f"Reward={ep_reward:7.1f}  "
                  f"PSNR={np.mean(ep_psnr):5.1f}  "
                  f"eps={epsilon:.3f}")

    return {'q_net': q_net, 'target_net': target_net, 'history': history, 'env': env}


print("Training function defined — starting training...")
print("="*60)
print("Training FULL agent (quality + temporal consistency)")
print("="*60)
result_full = train_agent(n_episodes=120, use_temporal_reward=True)

print()
print("="*60)
print("Training QUALITY-ONLY agent (no temporal reward)")
print("="*60)
result_qual = train_agent(n_episodes=120, use_temporal_reward=False)

## 9 — Training Curves

In [ ]:
def smooth(data, window=7):
    if len(data) < window:
        return data
    kernel = np.ones(window) / window
    return np.convolve(data, kernel, mode='valid')


fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Episode reward
ax = axes[0, 0]
ax.plot(smooth(result_full['history']['episode_reward']), 'b-', label='Full Agent', linewidth=2)
ax.plot(smooth(result_qual['history']['episode_reward']), 'r--', label='Quality-Only', linewidth=2)
ax.set_title('Episode Reward', fontsize=13, fontweight='bold')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Average PSNR
ax = axes[0, 1]
ax.plot(smooth(result_full['history']['avg_psnr']), 'b-', label='Full Agent', linewidth=2)
ax.plot(smooth(result_qual['history']['avg_psnr']), 'r--', label='Quality-Only', linewidth=2)
ax.set_title('Average PSNR per Episode', fontsize=13, fontweight='bold')
ax.set_xlabel('Episode')
ax.set_ylabel('PSNR (dB)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Temporal consistency
ax = axes[1, 0]
ax.plot(smooth(result_full['history']['avg_temporal']), 'b-', label='Full Agent', linewidth=2)
ax.plot(smooth(result_qual['history']['avg_temporal']), 'r--', label='Quality-Only', linewidth=2)
ax.set_title('Temporal Consistency (Higher = Better)', fontsize=13, fontweight='bold')
ax.set_xlabel('Episode')
ax.set_ylabel('Avg $R_{temporal}$')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Loss
ax = axes[1, 1]
ax.plot(smooth(result_full['history']['loss']), 'b-', label='Full Agent', linewidth=2)
ax.plot(smooth(result_qual['history']['loss']), 'r--', label='Quality-Only', linewidth=2)
ax.set_title('Training Loss (Huber)', fontsize=13, fontweight='bold')
ax.set_xlabel('Episode')
ax.set_ylabel('Loss')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

fig.suptitle('Training Dynamics — Full Agent vs Quality-Only Agent', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 10 — Evaluation & Results

We evaluate both agents on fresh synthetic sequences and compare
per-frame quality, temporal consistency, and flickering.

In [ ]:
def evaluate_agent(
    q_net: RecurrentDQN,
    clean_seq: np.ndarray,
    degraded_seq: np.ndarray,
    use_temporal_reward: bool = True,
) -> Dict:
    """Run a greedy evaluation pass and collect per-frame metrics."""
    env = VideoEnhancementEnv()
    if not use_temporal_reward:
        env.beta = 0.0
        env.gamma_a = 0.0
        env.lam_flicker = 0.0

    state = env.reset(clean_seq, degraded_seq)
    hidden = torch.zeros(1, q_net.hidden_size, device=DEVICE)
    done = False
    actions = []

    while not done:
        with torch.no_grad():
            s_t = torch.from_numpy(state).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
            q_vals, h_new = q_net(s_t, hidden)
            action = int(q_vals.argmax(dim=1).item())
        state, _, done, info = env.step(action)
        hidden = h_new
        actions.append(action)

    enhanced = np.array(env.enhanced_frames)

    psnr_degraded = [compute_psnr(degraded_seq[i], clean_seq[i]) for i in range(len(clean_seq))]
    psnr_enhanced = [compute_psnr(enhanced[i], clean_seq[i]) for i in range(len(enhanced))]

    temp_degraded, temp_enhanced = [], []
    for i in range(1, len(clean_seq)):
        f1d, f2d = frame_features(degraded_seq[i-1]), frame_features(degraded_seq[i])
        temp_degraded.append(np.linalg.norm(f1d - f2d))
    for i in range(1, len(enhanced)):
        f1e, f2e = frame_features(enhanced[i-1]), frame_features(enhanced[i])
        temp_enhanced.append(np.linalg.norm(f1e - f2e))

    flicker_deg = [abs(degraded_seq[i].mean() - 2*degraded_seq[i-1].mean() + degraded_seq[max(i-2,0)].mean())
                   for i in range(2, len(degraded_seq))]
    flicker_enh = [abs(enhanced[i].mean() - 2*enhanced[i-1].mean() + enhanced[max(i-2,0)].mean())
                   for i in range(2, len(enhanced))]

    return {
        'enhanced': enhanced,
        'actions': actions,
        'psnr_degraded': psnr_degraded,
        'psnr_enhanced': psnr_enhanced,
        'temp_degraded': temp_degraded,
        'temp_enhanced': temp_enhanced,
        'flicker_degraded': flicker_deg,
        'flicker_enhanced': flicker_enh,
    }


rng_eval = np.random.RandomState(999)
eval_clean = generate_clean_sequence(rng=rng_eval)
eval_degraded = degrade_sequence(eval_clean, rng=np.random.RandomState(1000))

eval_full = evaluate_agent(result_full['q_net'], eval_clean, eval_degraded, use_temporal_reward=True)
eval_qual = evaluate_agent(result_qual['q_net'], eval_clean, eval_degraded, use_temporal_reward=False)

print(f"Full Agent  — mean PSNR: {np.mean(eval_full['psnr_enhanced']):.2f} dB  "
      f"(degraded: {np.mean(eval_full['psnr_degraded']):.2f} dB)")
print(f"Qual Agent  — mean PSNR: {np.mean(eval_qual['psnr_enhanced']):.2f} dB  "
      f"(degraded: {np.mean(eval_qual['psnr_degraded']):.2f} dB)")
print(f"Full Agent  — mean flicker: {np.mean(eval_full['flicker_enhanced']):.4f}  "
      f"(degraded: {np.mean(eval_full['flicker_degraded']):.4f})")
print(f"Qual Agent  — mean flicker: {np.mean(eval_qual['flicker_enhanced']):.4f}  "
      f"(degraded: {np.mean(eval_qual['flicker_degraded']):.4f})")

### 10.1 Filmstrip View — Before / After Enhancement

In [ ]:
n_show = min(10, NUM_FRAMES)
show_idx = np.linspace(0, NUM_FRAMES - 1, n_show, dtype=int)

fig, axes = plt.subplots(3, n_show, figsize=(20, 7))
for col, idx in enumerate(show_idx):
    axes[0, col].imshow(eval_clean[idx], cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f't={idx}', fontsize=9)
    axes[0, col].axis('off')

    axes[1, col].imshow(eval_degraded[idx], cmap='gray', vmin=0, vmax=1)
    axes[1, col].axis('off')

    if idx < len(eval_full['enhanced']):
        axes[2, col].imshow(eval_full['enhanced'][idx], cmap='gray', vmin=0, vmax=1)
    axes[2, col].axis('off')

row_labels = ['Clean (GT)', 'Degraded', 'Enhanced (Full Agent)']
for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=11, rotation=90, labelpad=40)

fig.suptitle('Filmstrip View — Frame-by-Frame Enhancement', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 10.2 Per-Frame Quality Metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
frames_x = np.arange(NUM_FRAMES)
ax.plot(frames_x, eval_full['psnr_degraded'], 'r--s', label='Degraded', markersize=5, linewidth=1.5)
ax.plot(frames_x, eval_full['psnr_enhanced'], 'b-o', label='Enhanced (Full)', markersize=5, linewidth=2)
ax.plot(frames_x, eval_qual['psnr_enhanced'], 'g-^', label='Enhanced (Qual-Only)', markersize=5, linewidth=1.5)
ax.set_xlabel('Frame Index', fontsize=12)
ax.set_ylabel('PSNR (dB)', fontsize=12)
ax.set_title('Per-Frame PSNR Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

ax = axes[1]
frames_t = np.arange(1, NUM_FRAMES)
ax.plot(frames_t, eval_full['temp_degraded'], 'r--s', label='Degraded', markersize=5, linewidth=1.5)
ax.plot(frames_t, eval_full['temp_enhanced'], 'b-o', label='Enhanced (Full)', markersize=5, linewidth=2)
ax.plot(frames_t, eval_qual['temp_enhanced'], 'g-^', label='Enhanced (Qual-Only)', markersize=5, linewidth=1.5)
ax.set_xlabel('Frame Index', fontsize=12)
ax.set_ylabel('Feature Distance (lower = more consistent)', fontsize=11)
ax.set_title('Temporal Consistency per Frame', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 10.3 Flickering Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Mean intensity over time
ax = axes[0]
ax.plot([f.mean() for f in eval_clean], 'g-o', label='Clean', markersize=5, linewidth=2)
ax.plot([f.mean() for f in eval_degraded], 'r--s', label='Degraded', markersize=5, linewidth=1.5)
ax.plot([f.mean() for f in eval_full['enhanced']], 'b-^', label='Enhanced (Full)', markersize=5, linewidth=2)
ax.plot([f.mean() for f in eval_qual['enhanced']], 'm-d', label='Enhanced (Qual-Only)', markersize=5, linewidth=1.5)
ax.set_xlabel('Frame Index', fontsize=12)
ax.set_ylabel('Mean Intensity', fontsize=12)
ax.set_title('Mean Frame Intensity Over Time', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Flicker metric
ax = axes[1]
frames_f = np.arange(2, NUM_FRAMES)
ax.bar(frames_f - 0.25, eval_full['flicker_degraded'], width=0.25, color='red', alpha=0.7, label='Degraded')
ax.bar(frames_f, eval_full['flicker_enhanced'], width=0.25, color='blue', alpha=0.7, label='Enhanced (Full)')
ax.bar(frames_f + 0.25, eval_qual['flicker_enhanced'], width=0.25, color='green', alpha=0.7, label='Enhanced (Qual-Only)')
ax.set_xlabel('Frame Index', fontsize=12)
ax.set_ylabel('Flicker Metric $|\\bar{I}_t - 2\\bar{I}_{t-1} + \\bar{I}_{t-2}|$', fontsize=11)
ax.set_title('Flickering Analysis (Lower = Smoother)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 10.4 Action Sequence Heatmap

This heatmap shows which actions the agent selects for each processing step
across all frames — revealing temporal patterns in its enhancement strategy.

In [ ]:
def build_action_grid(actions: List[int], steps_per_frame: int, n_frames: int) -> np.ndarray:
    """Reshape flat action list into (n_frames, steps_per_frame) grid."""
    total_needed = n_frames * steps_per_frame
    acts = actions[:total_needed]
    if len(acts) < total_needed:
        acts = acts + [NUM_ACTIONS - 1] * (total_needed - len(acts))
    return np.array(acts).reshape(n_frames, steps_per_frame)


fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, eval_data, title in [
    (axes[0], eval_full, 'Full Agent'),
    (axes[1], eval_qual, 'Quality-Only Agent'),
]:
    grid = build_action_grid(eval_data['actions'], 3, NUM_FRAMES)
    im = ax.imshow(grid.T, aspect='auto', cmap='tab10', vmin=0, vmax=NUM_ACTIONS - 1,
                   interpolation='nearest')
    ax.set_xlabel('Frame Index', fontsize=12)
    ax.set_ylabel('Step within Frame', fontsize=12)
    ax.set_title(f'Action Heatmap — {title}', fontsize=13, fontweight='bold')
    ax.set_yticks(range(3))
    ax.set_yticklabels(['Step 1', 'Step 2', 'Step 3'])
    ax.set_xticks(range(0, NUM_FRAMES, 2))

cbar = fig.colorbar(im, ax=axes, ticks=range(NUM_ACTIONS), shrink=0.9)
cbar.ax.set_yticklabels(ACTION_NAMES, fontsize=9)
cbar.set_label('Action', fontsize=11)

plt.tight_layout()
plt.show()

### 10.5 Comparison — With vs Without Temporal Consistency Reward

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.35, wspace=0.3)

# Bar chart — average metrics
ax = fig.add_subplot(gs[0, :])
labels = ['PSNR (dB)', 'Temporal Consistency\n(lower = better)', 'Flicker\n(lower = better)']
full_vals = [
    np.mean(eval_full['psnr_enhanced']),
    np.mean(eval_full['temp_enhanced']),
    np.mean(eval_full['flicker_enhanced']),
]
qual_vals = [
    np.mean(eval_qual['psnr_enhanced']),
    np.mean(eval_qual['temp_enhanced']),
    np.mean(eval_qual['flicker_enhanced']),
]
deg_vals = [
    np.mean(eval_full['psnr_degraded']),
    np.mean(eval_full['temp_degraded']),
    np.mean(eval_full['flicker_degraded']),
]
x_pos = np.arange(len(labels))
width = 0.25
bars1 = ax.bar(x_pos - width, deg_vals, width, label='Degraded', color='red', alpha=0.7)
bars2 = ax.bar(x_pos, full_vals, width, label='Full Agent', color='blue', alpha=0.7)
bars3 = ax.bar(x_pos + width, qual_vals, width, label='Quality-Only', color='green', alpha=0.7)
ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=11)
ax.set_title('Overall Comparison: Degraded vs Full vs Quality-Only', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.01 * max(abs(v) for v in deg_vals+full_vals+qual_vals),
                f'{h:.2f}', ha='center', va='bottom', fontsize=8)

# Filmstrip: Full agent
n_strip = min(8, NUM_FRAMES)
strip_idx = np.linspace(0, NUM_FRAMES - 1, n_strip, dtype=int)
for col_i, idx in enumerate(strip_idx):
    ax_f = fig.add_subplot(gs[1, 0]) if col_i == 0 else None

ax_full = fig.add_subplot(gs[1, 0])
strip_full = np.concatenate([eval_full['enhanced'][i] for i in strip_idx], axis=1)
ax_full.imshow(strip_full, cmap='gray', vmin=0, vmax=1, aspect='auto')
ax_full.set_title('Full Agent — Enhanced Filmstrip', fontsize=12, fontweight='bold')
ax_full.set_xticks(np.arange(n_strip) * IMG_W + IMG_W // 2)
ax_full.set_xticklabels([f't={i}' for i in strip_idx], fontsize=8)
ax_full.set_yticks([])

ax_qual = fig.add_subplot(gs[1, 1])
strip_qual = np.concatenate([eval_qual['enhanced'][i] for i in strip_idx], axis=1)
ax_qual.imshow(strip_qual, cmap='gray', vmin=0, vmax=1, aspect='auto')
ax_qual.set_title('Quality-Only Agent — Enhanced Filmstrip', fontsize=12, fontweight='bold')
ax_qual.set_xticks(np.arange(n_strip) * IMG_W + IMG_W // 2)
ax_qual.set_xticklabels([f't={i}' for i in strip_idx], fontsize=8)
ax_qual.set_yticks([])

# Degraded filmstrip for reference
ax_deg = fig.add_subplot(gs[2, 0])
strip_deg = np.concatenate([eval_degraded[i] for i in strip_idx], axis=1)
ax_deg.imshow(strip_deg, cmap='gray', vmin=0, vmax=1, aspect='auto')
ax_deg.set_title('Degraded Input — Filmstrip', fontsize=12, fontweight='bold')
ax_deg.set_xticks(np.arange(n_strip) * IMG_W + IMG_W // 2)
ax_deg.set_xticklabels([f't={i}' for i in strip_idx], fontsize=8)
ax_deg.set_yticks([])

ax_clean = fig.add_subplot(gs[2, 1])
strip_clean = np.concatenate([eval_clean[i] for i in strip_idx], axis=1)
ax_clean.imshow(strip_clean, cmap='gray', vmin=0, vmax=1, aspect='auto')
ax_clean.set_title('Clean Ground Truth — Filmstrip', fontsize=12, fontweight='bold')
ax_clean.set_xticks(np.arange(n_strip) * IMG_W + IMG_W // 2)
ax_clean.set_xticklabels([f't={i}' for i in strip_idx], fontsize=8)
ax_clean.set_yticks([])

fig.suptitle('Full Comparison Dashboard', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 10.6 Action Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, eval_data, title, color in [
    (axes[0], eval_full, 'Full Agent', 'steelblue'),
    (axes[1], eval_qual, 'Quality-Only Agent', 'seagreen'),
]:
    counts = np.bincount(eval_data['actions'], minlength=NUM_ACTIONS)
    pct = counts / counts.sum() * 100
    bars = ax.barh(range(NUM_ACTIONS), pct, color=color, alpha=0.8)
    ax.set_yticks(range(NUM_ACTIONS))
    ax.set_yticklabels(ACTION_NAMES, fontsize=10)
    ax.set_xlabel('Usage (%)', fontsize=12)
    ax.set_title(f'Action Distribution — {title}', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    for bar, p in zip(bars, pct):
        if p > 0:
            ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                    f'{p:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

### 10.7 Detailed Frame-by-Frame Enhancement Gallery

In [ ]:
gallery_idx = [0, 5, 10, 15, 19]
fig, axes = plt.subplots(4, len(gallery_idx), figsize=(18, 12))

row_titles = ['Clean (GT)', 'Degraded', 'Enhanced (Full)', 'Enhancement Δ']

for col, idx in enumerate(gallery_idx):
    axes[0, col].imshow(eval_clean[idx], cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f'Frame {idx}\nPSNR ref', fontsize=10)
    axes[0, col].axis('off')

    axes[1, col].imshow(eval_degraded[idx], cmap='gray', vmin=0, vmax=1)
    psnr_d = compute_psnr(eval_degraded[idx], eval_clean[idx])
    axes[1, col].set_title(f'PSNR: {psnr_d:.1f} dB', fontsize=10)
    axes[1, col].axis('off')

    if idx < len(eval_full['enhanced']):
        enh = eval_full['enhanced'][idx]
        axes[2, col].imshow(enh, cmap='gray', vmin=0, vmax=1)
        psnr_e = compute_psnr(enh, eval_clean[idx])
        axes[2, col].set_title(f'PSNR: {psnr_e:.1f} dB', fontsize=10, color='blue')
        axes[2, col].axis('off')

        diff = np.abs(enh.astype(float) - eval_degraded[idx].astype(float))
        axes[3, col].imshow(diff, cmap='hot', vmin=0, vmax=0.3)
        axes[3, col].set_title(f'Max Δ: {diff.max():.3f}', fontsize=10)
        axes[3, col].axis('off')
    else:
        axes[2, col].axis('off')
        axes[3, col].axis('off')

for row, label in enumerate(row_titles):
    axes[row, 0].set_ylabel(label, fontsize=11, rotation=90, labelpad=40)

fig.suptitle('Detailed Frame Enhancement Gallery', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 11 — Summary

### What We Built

A complete **RL-based video frame enhancement pipeline** that processes frames
sequentially with temporal memory, maintaining visual consistency across a clip.

### Key Takeaways

| Topic | Detail |
|-------|--------|
| **State design** | Current frame + GRU hidden state encodes temporal context: $s_t = [I_t, h_{t-1}]$ |
| **Action space** | 10 discrete operations including temporal-aware actions (smooth, colour-correct) |
| **Reward engineering** | Multi-objective: quality ($\text{PSNR}$), temporal consistency, action smoothness, flicker penalty |
| **Architecture** | Recurrent DQN with CNN feature extractor + GRU temporal cell |
| **Temporal consistency** | Explicit reward terms reduce flickering compared to quality-only optimisation |
| **Flicker penalty** | Second-order temporal derivative penalty $|\bar{I}_t - 2\bar{I}_{t-1} + \bar{I}_{t-2}|$ |

### Mathematical Highlights

- **Combined reward:** $R_t = \alpha \cdot R_{\text{quality}} + \beta \cdot R_{\text{temporal}} - \gamma_a \cdot \|a_t - a_{t-1}\| - \lambda \cdot \text{Flicker}_t$
- **Recurrent Q-learning:** $Q(s_t, a; \theta) = W_q \;\text{GRU}(\phi(I_t), h_{t-1}) + b_q$
- **Temporal feature distance:** $d_t = \|f(I_t^{\text{enh}}) - f(I_{t-1}^{\text{enh}})\|_2$

### Extensions

- Replace GRU with **Transformer attention** over a sliding window of past frames
- Use **continuous actions** (PPO/SAC) for finer-grained parameter control
- Add **optical-flow-based** scene-change detection for smarter temporal gating
- Scale to **colour video** with per-channel enhancement actions
- Incorporate **perceptual losses** (LPIPS) for more human-aligned quality metrics

---

*Module 12.4 — Reinforcement Learning for Image Processing*